In [171]:
import pandas as pd
import math as m
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score,classification_report
from sklearn.neighbors import KNeighborsClassifier

In [172]:
df=pd.read_csv('fake_bills.csv',sep=';')
df.head()


,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length
0,True,171.81,104.86,104.95,4.52,2.89,112.83
1,True,171.46,103.36,103.66,3.77,2.99,113.09
2,True,172.69,104.48,103.50,4.40,2.94,113.16
3,True,171.36,103.91,103.94,3.62,3.01,113.51
4,True,171.73,104.28,103.46,4.04,3.48,112.54


In [173]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   is_genuine    1500 non-null   bool   
 1   diagonal      1500 non-null   float64
 2   height_left   1500 non-null   float64
 3   height_right  1500 non-null   float64
 4   margin_low    1463 non-null   float64
 5   margin_up     1500 non-null   float64
 6   length        1500 non-null   float64
dtypes: bool(1), float64(6)
memory usage: 71.9 KB


In [174]:
df.duplicated().sum()

0

In [175]:
df.isnull().sum()

is_genuine       0
diagonal         0
height_left      0
height_right     0
margin_low      37
margin_up        0
length           0
dtype: int64

In [176]:
imputer=SimpleImputer(strategy='mean')
df['margin_low']=imputer.fit_transform(df[['margin_low']])

In [177]:
df.isnull().sum()

is_genuine      0
diagonal        0
height_left     0
height_right    0
margin_low      0
margin_up       0
length          0
dtype: int64

In [178]:
x=df.drop('is_genuine',axis=1)
y=df['is_genuine']

In [179]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [180]:
num_col=df.select_dtypes(include='number').columns

In [181]:
df.describe()

,diagonal,height_left,height_right,margin_low,margin_up,length
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.00000
mean,171.958440,104.029533,103.920307,4.485967,3.151473,112.67850
std,0.305195,0.299462,0.325627,0.655569,0.231813,0.87273
min,171.040000,103.140000,102.820000,2.980000,2.270000,109.49000
25%,171.750000,103.820000,103.710000,4.030000,2.990000,112.03000
50%,171.960000,104.040000,103.920000,4.330000,3.140000,112.96000
75%,172.170000,104.230000,104.150000,4.860000,3.310000,113.34000
max,173.010000,104.880000,104.950000,6.900000,3.910000,114.44000


In [182]:
preprocess=ColumnTransformer(transformers=[
    ('scale',RobustScaler(),num_col)
])
preprocess.fit_transform(xtrain,ytrain)

array([[ 0.76190476,  1.41463415, -0.27118644, -0.30670927, -0.15625   ,
         0.21663443],
       [-0.14285714,  1.12195122, -0.97175141, -0.30670927, -0.96875   ,
         0.40232108],
       [-0.78571429,  0.70731707,  1.55932203,  0.71565495,  0.6875    ,
        -0.6344294 ],
       ...,
       [-1.38095238, -0.51219512,  0.15819209,  0.15335463, -0.03125   ,
         0.3868472 ],
       [-0.42857143,  0.65853659, -0.2259887 ,  2.38977636,  0.375     ,
        -1.42359768],
       [-0.28571429,  0.04878049,  0.40677966,  0.7028754 ,  0.375     ,
        -1.50096712]])

In [183]:
pipline=Pipeline(steps=[
    ('preprocess',preprocess),
    ('model',KNeighborsClassifier())
])
pipline.fit(xtrain,ytrain)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('scale', RobustScaler(),
                                                  Index(['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up',
       'length'],
      dtype='object'))])),
                ('model', KNeighborsClassifier())])

In [184]:
k=m.sqrt(1500)
k

38.72983346207417

In [185]:
cv=GridSearchCV(estimator=pipline,param_grid={
    'model__n_neighbors':[5,38,37,39,40],
    'model__weights':['uniform','distance'],
    'model__metric':['euclidean','manhattan']
    },cv=10,scoring="f1")
cv.fit(xtrain,ytrain)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('scale',
                                                                         RobustScaler(),
                                                                         Index(['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up',
       'length'],
      dtype='object'))])),
                                       ('model', KNeighborsClassifier())]),
             param_grid={'model__metric': ['euclidean', 'manhattan'],
                         'model__n_neighbors': [5, 38, 37, 39, 40],
                         'model__weights': ['uniform', 'distance']},
             scoring='f1')

In [186]:
cv.best_estimator_

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('scale', RobustScaler(),
                                                  Index(['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up',
       'length'],
      dtype='object'))])),
                ('model', KNeighborsClassifier(metric='manhattan'))])

In [187]:
model=cv.best_params_

In [188]:
pd.DataFrame(cv.cv_results_).sort_values('rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__metric,param_model__n_neighbors,param_model__weights,params,split0_test_score,split1_test_score,...,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
11,0.007703,0.006585,0.009788,0.006067,manhattan,5,distance,"{'model__metric': 'manhattan', 'model__n_neigh...",1.000000,0.987805,...,0.993865,0.987805,0.987805,0.981366,0.993865,1.000000,0.987805,0.991418,0.005644,1
10,0.009538,0.003773,0.015220,0.004051,manhattan,5,uniform,"{'model__metric': 'manhattan', 'model__n_neigh...",1.000000,0.987805,...,0.993865,0.987805,0.987805,0.981366,0.993865,1.000000,0.987805,0.991418,0.005644,1
0,0.015556,0.014379,0.022312,0.013817,euclidean,5,uniform,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.987654,...,1.000000,0.987805,0.987805,0.981366,0.993865,1.000000,0.987805,0.991395,0.006289,3
1,0.007890,0.000927,0.005857,0.003113,euclidean,5,distance,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.987654,...,1.000000,0.987805,0.987805,0.981366,0.993865,1.000000,0.987805,0.991395,0.006289,3
2,0.006099,0.005428,0.006666,0.004963,euclidean,38,uniform,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.993865,...,0.993865,0.981818,0.987654,0.987654,0.993865,0.993865,0.981818,0.990827,0.005592,5
3,0.005157,0.006087,0.006565,0.008070,euclidean,38,distance,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.993865,...,0.993865,0.981818,0.987654,0.987654,0.993865,0.993865,0.981818,0.990827,0.005592,5
4,0.001052,0.003156,0.014935,0.005237,euclidean,37,uniform,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.993865,...,0.993865,0.981818,0.987654,0.987654,0.993865,0.993865,0.981818,0.990827,0.005592,5
6,0.013250,0.004041,0.020151,0.009003,euclidean,39,uniform,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.993865,...,0.993865,0.981818,0.987654,0.987654,0.993865,0.993865,0.981818,0.990827,0.005592,5
9,0.012385,0.004956,0.015661,0.002505,euclidean,40,distance,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.993865,...,0.993865,0.981818,0.987654,0.987654,0.993865,0.993865,0.981818,0.990206,0.005565,9
7,0.012365,0.004060,0.010008,0.003470,euclidean,39,distance,"{'model__metric': 'euclidean', 'model__n_neigh...",1.000000,0.993865,...,0.993865,0.981818,0.987654,0.987654,0.993865,0.993865,0.981818,0.990206,0.005565,9


In [191]:
ypred_xtrain=cv.predict(xtrain)
ypred_xtest=cv.predict(xtest)

In [193]:
print(classification_report(ytrain,ypred_xtrain))

              precision    recall  f1-score   support

       False       0.99      0.97      0.98       390
        True       0.99      1.00      0.99       810

    accuracy                           0.99      1200
   macro avg       0.99      0.99      0.99      1200
weighted avg       0.99      0.99      0.99      1200



In [194]:
print(classification_report(ytest,ypred_xtest))

              precision    recall  f1-score   support

       False       1.00      0.97      0.99       110
        True       0.98      1.00      0.99       190

    accuracy                           0.99       300
   macro avg       0.99      0.99      0.99       300
weighted avg       0.99      0.99      0.99       300

